[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C00_VLM_Multimodal_Course/04_connectors_training/04_connectors.ipynb)

# 04 · 连接器与训练范式 —— 从零实现三种 Connector

配套讲解：`04_讲解.html`。本 notebook 的重点是**从零用 PyTorch 实现 VLM 连接器（connector）**，并把它们串到真实 CLIP 视觉塔上，亲手量化"token 数 vs 细节"这条贯穿全课的权衡。

我们会实现并对比三种连接器：

| 连接器 | 思路 | token 数变化 |
|---|---|---|
| **MLP projector**（LLaVA-1.5） | 逐 token 投影 `Linear→GELU→Linear` | 不变（576→576） |
| **Q-Former 风格压缩器**（BLIP-2） | learnable queries 对图像 cross-attn | 压成固定 32 |
| **Perceiver Resampler**（Flamingo） | learnable latents 重采样，长度无关 | 压成固定 64 |

最后做一个 **stage-1 对齐 plumbing 演示**：冻结 `gpt2` 词嵌入，只训 MLP projector 几步，看 loss 下降——展示"只训 connector"的对齐阶段如何工作（**教学简化版**，不是真实 caption 语言建模）。

**算力**：核心连接器实验纯 CPU 秒级即可。加载 `openai/clip-vit-base-patch32`（约 600MB）那一格需联网首次下载，CPU 也能跑（单图推理 <1s）。"GPU 可选"。

正文之后附 **3 道 ✏️ 练习**（numpy 纯 CPU、每格可独立运行）+ 📖 参考答案。

## 1. 环境与设备

第一格统一：import + 选择设备 + 打印版本。CPU 即可完成全部内容。

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("torch :", torch.__version__)
print("device:", device)
torch.manual_seed(0)  # 结果可复现

torch : 2.11.0+cu128
device: cuda


## 2. MLP projector（LLaVA-1.5 风格）—— 维度对齐，token 数不变

最简单的连接器：对**每个** patch token 独立做投影，把视觉维度 `d_v` 映到 LLM 维度 `d_llm`，**token 数原封不动**。

LLaVA-1.5 的结构正是 `Linear(d_v, d_llm) → GELU → Linear(d_llm, d_llm)`（两层 MLP + GELU）。我们用一批随机/真实的 patch 特征 `[B, 576, 1024]`（CLIP ViT-L/14@336 的输出形状），打印投影后的 `[B, 576, 4096]`（Vicuna-7B 隐藏维），确认 **token 数 576 不变**。

In [4]:
class MLPProjector(nn.Module):
    '''LLaVA-1.5 风格连接器：逐 token 投影，token 数不变。'''
    def __init__(self, d_v=1024, d_llm=4096):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(d_v, d_llm),
            nn.GELU(),
            nn.Linear(d_llm, d_llm),
        )

    def forward(self, x):           # x: [B, N, d_v]
        return self.proj(x)         # -> [B, N, d_llm]，N 不变


B, N, d_v, d_llm = 2, 576, 1024, 4096
patch_feats = torch.randn(B, N, d_v)        # 模拟 CLIP ViT-L/14@336 输出
mlp = MLPProjector(d_v, d_llm)

out = mlp(patch_feats)
print("输入  patch 特征:", tuple(patch_feats.shape))   # (2, 576, 1024)
print("输出  视觉 token:", tuple(out.shape))           # (2, 576, 4096)
print("token 数:", patch_feats.shape[1], "->", out.shape[1], "（不变，保留全部 patch）")
print("可训练参数量: {:,}".format(sum(p.numel() for p in mlp.parameters())))

输入  patch 特征: (2, 576, 1024)
输出  视觉 token: (2, 576, 4096)
token 数: 576 -> 576 （不变，保留全部 patch）
可训练参数量: 20,979,712


**看输出哪里**：token 维（dim=1）保持 576 不变，只有特征维从 1024 变成 4096。MLP projector 的优点正在于此——保留全部细粒度 patch，对 OCR/文档/定位友好；代价是 576 个视觉 token 会占满 LLM 的上下文预算。

## 3. Q-Former 风格压缩器（BLIP-2）—— 压成固定 32 token

Q-Former 的核心：一组**可学习的 query**（`nn.Parameter`，BLIP-2 用 32 个）通过 **cross-attention** 主动从图像特征里"问出"信息。无论输入多少 patch，输出永远是 32 个 token。

这里用 `nn.MultiheadAttention`（`batch_first=True`）实现最简版：`query=32 个 learnable queries`，`key/value=图像特征`。真实 Q-Former 还有 query 间的 self-attention 与 ITC/ITM/ITG 预训练目标（见讲解第 3 节），此处聚焦"固定长度压缩"这一核心机制。

In [5]:
class QFormerCompressor(nn.Module):
    '''最简 Q-Former 风格压缩器：learnable queries 对图像特征做 cross-attn。'''
    def __init__(self, d_v=1024, num_query=32, n_heads=8):
        super().__init__()
        # 32 个可学习 query，形状 [num_query, d_v]
        self.queries = nn.Parameter(torch.randn(num_query, d_v) * 0.02)
        self.self_attn  = nn.MultiheadAttention(d_v, n_heads, batch_first=True)   # query 间交流
        self.cross_attn = nn.MultiheadAttention(d_v, n_heads, batch_first=True)   # query <- 图像
        self.ln1 = nn.LayerNorm(d_v)
        self.ln2 = nn.LayerNorm(d_v)

    def forward(self, image_feats):                 # image_feats: [B, N, d_v]
        B = image_feats.size(0)
        q = self.queries.unsqueeze(0).expand(B, -1, -1)   # [B, 32, d_v]
        # query 之间 self-attention
        sa, _ = self.self_attn(q, q, q)
        q = self.ln1(q + sa)
        # query(Q) 对图像特征(K,V) cross-attention —— 信息瓶颈在这里
        ca, attn_w = self.cross_attn(q, image_feats, image_feats)
        q = self.ln2(q + ca)
        return q, attn_w                            # [B, 32, d_v]


img_feats = torch.randn(B, N, d_v)             # [2, 576, 1024]
qformer = QFormerCompressor(d_v=d_v, num_query=32)

q_out, attn_w = qformer(img_feats)
print("输入图像特征:", tuple(img_feats.shape))     # (2, 576, 1024)
print("输出 query  :", tuple(q_out.shape))        # (2, 32, 1024)
print("token 数:", img_feats.shape[1], "->", q_out.shape[1], "（576 压成固定 32）")
print("cross-attn 权重形状:", tuple(attn_w.shape), "= [B, 32 个query, 576 个patch]")
print("每个 query 的注意力权重和 ~", attn_w[0, 0].sum().item(), "（softmax 归一，应≈1）")

输入图像特征: (2, 576, 1024)
输出 query  : (2, 32, 1024)
token 数: 576 -> 32 （576 压成固定 32）
cross-attn 权重形状: (2, 32, 576) = [B, 32 个query, 576 个patch]
每个 query 的注意力权重和 ~ 1.0 （softmax 归一，应≈1）


**看输出哪里**：输出是固定的 32 个 token，与输入的 576 个 patch 解耦。`attn_w[b, i, :]` 是第 `i` 个 query 在 576 个 patch 上的注意力分布（softmax 归一，和≈1），可视化它能看到"每个 query 关注图像哪些区域"。强压缩省上下文，但 32 token 是信息瓶颈——高分辨率密集文字会受损。

## 4. Perceiver Resampler（Flamingo）—— 长度无关，固定输出 64

Perceiver Resampler 与 Q-Former 同属"learnable latent + cross-attn"家族，但更强调**长度无关（length-agnostic）**：无论输入 49 个还是 256 个 patch，输出永远是 `R=64` 个 latent token。

Flamingo 的一个设计细节：cross-attn 的 K/V 是 **`[latents; 视觉特征]` 拼接**（让 latent 也能看到自己）。下面验证"输入变长 → 输出固定"。

In [6]:
class PerceiverResampler(nn.Module):
    '''Flamingo 风格重采样器：固定数量 latent，长度无关。'''
    def __init__(self, d_v=1024, num_latents=64, n_heads=8):
        super().__init__()
        self.latents = nn.Parameter(torch.randn(num_latents, d_v) * 0.02)
        self.cross_attn = nn.MultiheadAttention(d_v, n_heads, batch_first=True)
        self.ln_q  = nn.LayerNorm(d_v)
        self.ln_kv = nn.LayerNorm(d_v)
        self.ff = nn.Sequential(nn.Linear(d_v, 4 * d_v), nn.GELU(), nn.Linear(4 * d_v, d_v))
        self.ln_ff = nn.LayerNorm(d_v)

    def forward(self, x):                           # x: [B, N, d_v]，N 可变
        B = x.size(0)
        lat = self.latents.unsqueeze(0).expand(B, -1, -1)        # [B, 64, d_v]
        # Flamingo: K/V = 拼接 [latents; 视觉特征]
        kv = torch.cat([lat, x], dim=1)
        q = self.ln_q(lat)
        ca, _ = self.cross_attn(q, self.ln_kv(kv), self.ln_kv(kv))
        lat = lat + ca
        lat = lat + self.ff(self.ln_ff(lat))
        return lat                                  # [B, 64, d_v]


resampler = PerceiverResampler(d_v=d_v, num_latents=64)

for n_patch in [49, 256]:                           # 两种不同输入长度
    x = torch.randn(B, n_patch, d_v)
    y = resampler(x)
    print(f"输入 {n_patch:>3} patch  ->  输出 {y.shape[1]} latent token   shape={tuple(y.shape)}")

print("\n结论：输入长度 49 vs 256 都映射到固定 64 个 token —— 长度无关，适合多帧视频/交错图文")

输入  49 patch  ->  输出 64 latent token   shape=(2, 64, 1024)
输入 256 patch  ->  输出 64 latent token   shape=(2, 64, 1024)

结论：输入长度 49 vs 256 都映射到固定 64 个 token —— 长度无关，适合多帧视频/交错图文


## 5. 加载真实 CLIP 视觉塔，三种连接器对照

现在用真实的 `openai/clip-vit-base-patch32` 取 patch 特征，分别过三种连接器，打印 **token 数对照表**，把"token 数 vs 细节"的权衡量化出来。

> ViT-B/32@224 产生 `(224/32)² = 49` 个 patch + 1 个 [CLS] = 50 个 token；我们取 49 个 patch（去掉 [CLS]）。
> **算力提示**：首次会下载约 600MB 权重，需联网；之后 CPU 单图推理 <1s。

In [7]:
from transformers import CLIPVisionModel, CLIPImageProcessor
from PIL import Image
import numpy as np

ckpt = "openai/clip-vit-base-patch32"
vision = CLIPVisionModel.from_pretrained(ckpt).to(device).eval()
processor = CLIPImageProcessor.from_pretrained(ckpt)

# 用一张随机彩色图占位（无需下载数据集）；换成真实图片同理
dummy = Image.fromarray((np.random.rand(224, 224, 3) * 255).astype("uint8"))
inputs = processor(images=dummy, return_tensors="pt").to(device)

with torch.no_grad():
    vout = vision(**inputs)

# last_hidden_state: [B, 1+num_patch, d_v]；第 0 个是 [CLS]，其余是 patch
last_hidden = vout.last_hidden_state          # [1, 50, 768]
d_v_real = last_hidden.shape[-1]
patch_tokens = last_hidden[:, 1:, :]          # 去掉 [CLS] -> [1, 49, 768]
print("CLIP last_hidden_state :", tuple(last_hidden.shape))
print("patch tokens (去[CLS]) :", tuple(patch_tokens.shape), " d_v =", d_v_real)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
logit_scale                                                  | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_at

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

CLIP last_hidden_state : (1, 50, 768)
patch tokens (去[CLS]) : (1, 49, 768)  d_v = 768


In [8]:
# 用真实 d_v=768 重新实例化三种连接器，目标 LLM 维度设 d_llm=4096（Vicuna-7B）
d_llm_real = 4096
mlp_r       = MLPProjector(d_v_real, d_llm_real)
qformer_r   = QFormerCompressor(d_v=d_v_real, num_query=32)
resampler_r = PerceiverResampler(d_v=d_v_real, num_latents=64)

x = patch_tokens.cpu()                        # [1, 49, 768]
with torch.no_grad():
    mlp_out         = mlp_r(x)                # [1, 49, 4096]
    qformer_out, _  = qformer_r(x)            # [1, 32, 768]
    perc_out        = resampler_r(x)          # [1, 64, 768]

print(f"{'连接器':<26}{'输入 token':<12}{'输出 token':<12}{'输出特征维'}")
print("-" * 62)
print(f"{'MLP projector (LLaVA)':<26}{x.shape[1]:<12}{mlp_out.shape[1]:<12}{mlp_out.shape[-1]}")
print(f"{'Q-Former (BLIP-2)':<26}{x.shape[1]:<12}{qformer_out.shape[1]:<12}{qformer_out.shape[-1]}")
print(f"{'Perceiver (Flamingo)':<26}{x.shape[1]:<12}{perc_out.shape[1]:<12}{perc_out.shape[-1]}")
print("\n权衡：MLP 保全部 token(细节全/占上下文) | Q-Former 压到 32(省/易丢细节) | Perceiver 固定 64(长度无关)")
print("注：MLP 已投到 d_llm=4096 可直接喂 LLM；Q-Former/Perceiver 还需一层 Linear(d_v->d_llm)")

连接器                       输入 token    输出 token    输出特征维
--------------------------------------------------------------
MLP projector (LLaVA)     49          49          4096
Q-Former (BLIP-2)         49          32          768
Perceiver (Flamingo)      49          64          768

权衡：MLP 保全部 token(细节全/占上下文) | Q-Former 压到 32(省/易丢细节) | Perceiver 固定 64(长度无关)
注：MLP 已投到 d_llm=4096 可直接喂 LLM；Q-Former/Perceiver 还需一层 Linear(d_v->d_llm)


**对照表读法**：同一张图，三种连接器给 LLM 的"视觉 token 账单"完全不同——49（保留）vs 32（强压）vs 64（重采样）。在 ViT-L/14@336 下 MLP 会是 576，差距更悬殊。讲解第 7 节的消融结论：在保留较多 token 且训练充分时，连接器**类型**对最终性能影响不大，真正的杠杆是分辨率与编码器。

## 6. Stage-1 对齐 plumbing 演示：冻结 LLM，只训 connector

两阶段训练的 stage-1：**冻结视觉编码器 + 冻结 LLM，只训 connector**，让它学会把视觉特征投到 LLM 的词嵌入空间。

下面用 `gpt2` 的词嵌入维度（`wte`，768 维）作为对齐目标：
1. 加载 `gpt2`，**冻结其全部参数**（`requires_grad_(False)`）；
2. 用一个 MLP projector 把 CLIP 特征（768）投到 gpt2 hidden（768）；
3. 构造一个对齐目标——让投影后的视觉 token 的**统计量靠近真实词嵌入的分布**（一个可优化的教学目标）；
4. 只优化 projector，跑几步，看 loss 下降。

> **这是教学简化版**：真实 stage-1 用海量"图-文 caption 对"做自回归语言建模 `-Σ log p(w_t | w_<t, g(v))`，需多卡。这里只演示 **plumbing**——梯度只流经 connector，验证"只训 connector"这件事本身如何工作。`gpt2` 约 124M 参数，CPU 可跑。

In [9]:
from transformers import GPT2Model

gpt2 = GPT2Model.from_pretrained("gpt2").to(device).eval()
gpt2.requires_grad_(False)                          # 冻结 LLM —— stage-1 关键
d_gpt = gpt2.config.n_embd                          # 768

# connector: CLIP 特征(768) -> gpt2 词嵌入空间(768)
connector = MLPProjector(d_v_real, d_gpt).to(device)

# 对齐目标：真实词嵌入的逐维均值/方差，作为"合理词嵌入分布"的代理统计量
wte = gpt2.wte.weight.detach()                      # [50257, 768]，已冻结
target_mean = wte.mean(0)                           # [768]
target_std  = wte.std(0)                            # [768]

# 检查：只有 connector 的参数 requires_grad
n_train = sum(p.numel() for p in connector.parameters() if p.requires_grad)
n_frozen = sum(p.numel() for p in gpt2.parameters())
print(f"可训练参数 (connector): {n_train:,}")
print(f"冻结参数   (gpt2 LLM ): {n_frozen:,}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

可训练参数 (connector): 1,181,184
冻结参数   (gpt2 LLM ): 124,439,808


In [10]:
# 准备一批 CLIP 视觉特征作为输入（这里复用第 5 节的 patch_tokens，扩成一个小 batch）
vis = patch_tokens.detach().to(device).repeat(4, 1, 1)    # [4, 49, 768]

opt = torch.optim.Adam(connector.parameters(), lr=1e-3)

print("step |   loss")
print("-----+---------")
for step in range(60):
    opt.zero_grad()
    proj = connector(vis)                           # [4, 49, 768] 投到 gpt2 空间
    # 对齐损失：让投影后视觉 token 的逐维均值/方差靠近真实词嵌入统计量
    m = proj.reshape(-1, d_gpt).mean(0)
    s = proj.reshape(-1, d_gpt).std(0)
    loss = F.mse_loss(m, target_mean) + F.mse_loss(s, target_std)
    loss.backward()
    opt.step()
    if step % 10 == 0 or step == 59:
        print(f"{step:4d} | {loss.item():.5f}")

print("\nloss 持续下降 = connector 正学着把视觉 token 投到接近 gpt2 词嵌入的分布。")
print("梯度只流经 connector（gpt2 冻结）——这正是 stage-1『只训 connector』的 plumbing。")

step |   loss
-----+---------
   0 | 0.02046
  10 | 0.00063
  20 | 0.00023
  30 | 0.00011
  40 | 0.00003
  50 | 0.00001
  59 | 0.00000

loss 持续下降 = connector 正学着把视觉 token 投到接近 gpt2 词嵌入的分布。
梯度只流经 connector（gpt2 冻结）——这正是 stage-1『只训 connector』的 plumbing。


In [11]:
# 验证：把对齐后的视觉 token 当作 gpt2 的 inputs_embeds 喂进冻结 LLM，能正常前向
with torch.no_grad():
    vis_embeds = connector(vis)                     # [4, 49, 768]
    llm_out = gpt2(inputs_embeds=vis_embeds)        # 把视觉 token 当成"前缀词嵌入"
print("视觉 token 作为 inputs_embeds 喂入冻结 gpt2 -> last_hidden_state:",
      tuple(llm_out.last_hidden_state.shape))
print("说明：connector 的输出已是 LLM 可消费的『词嵌入式』token（shallow fusion 前缀拼接的前一步）")

视觉 token 作为 inputs_embeds 喂入冻结 gpt2 -> last_hidden_state: (4, 49, 768)
说明：connector 的输出已是 LLM 可消费的『词嵌入式』token（shallow fusion 前缀拼接的前一步）


---
## ✏️ 练习 1：用 numpy 重写 MLP projector 前向

不翻上文，徒手实现 LLaVA-1.5 连接器的前向：`Linear(d_v→d_llm) → GELU → Linear(d_llm→d_llm)`。这次不用 `nn.Module`——权重直接作为参数传入，逼自己写出矩阵乘法本体（3 道练习都用 numpy 纯 CPU，每格可独立运行，不依赖上文）。

输入 `x: [B, N, d_v]`，参数 `W1: [d_v, d_llm]`、`b1: [d_llm]`、`W2: [d_llm, d_llm]`、`b2: [d_llm]`，返回 `[B, N, d_llm]`。

**提示**：`x @ W1 + b1` 在最后一维做投影，batch 维自动广播；GELU 用 tanh 近似 $0.5x\left(1+\tanh\big(\sqrt{2/\pi}\,(x+0.044715x^3)\big)\right)$；逐 token 投影意味着 token 之间**互不通信**——自测里会用"换序输入 = 换序输出"验证这一点。两个函数合计 5 行以内。

In [12]:
import numpy as np

def gelu(x):
    # TODO: tanh 近似版 GELU：0.5*x*(1+tanh(sqrt(2/pi)*(x+0.044715*x**3)))
    raise NotImplementedError

def mlp_projector(x, W1, b1, W2, b2):
    # TODO: x @ W1 + b1 -> GELU -> @ W2 + b2，返回 [B, N, d_llm]
    raise NotImplementedError

In [14]:
# 练习 1 参考答案（先自己做，再对照）
import numpy as np

def gelu(x):
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x ** 3)))

def mlp_projector(x, W1, b1, W2, b2):
    return gelu(x @ W1 + b1) @ W2 + b2

In [15]:
# —— 练习 1 自测 ——
import numpy as np
rng = np.random.default_rng(0)
B, N, d_v, d_llm = 2, 6, 8, 16
x  = rng.standard_normal((B, N, d_v))
W1 = rng.standard_normal((d_v, d_llm)) * 0.1
b1 = np.zeros(d_llm)
W2 = rng.standard_normal((d_llm, d_llm)) * 0.1
b2 = np.zeros(d_llm)

out = mlp_projector(x, W1, b1, W2, b2)
assert out.shape == (B, N, d_llm)                        # 维度对齐：d_v -> d_llm
assert out.shape[1] == x.shape[1]                        # token 数不变
assert abs(gelu(np.array(0.0))) < 1e-12                  # GELU(0) = 0
assert np.allclose(mlp_projector(np.zeros((1, 3, d_v)), W1, b1, W2, b2), 0.0)  # 零输入+零偏置 -> 零输出
perm = [3, 1, 5, 0, 2, 4]
assert np.allclose(mlp_projector(x[:, perm], W1, b1, W2, b2), out[:, perm])    # 逐 token 独立：换序输入 = 换序输出
print("✅ 练习 1 通过")

✅ 练习 1 通过


## ✏️ 练习 2：实现 average-pooling 降采样 connector

第四条压缩路线：不引入任何可学习参数，直接在 2D patch 网格上做 k×k 平均池化（MobileVLM 的 LDP、LLaVA-OneVision 等都用过这类廉价压缩）。实现 `avgpool_connector(x, k)`：输入 `x: [B, N, d]`（`N = g*g`，patch 已按行优先展平），先还原成 `[B, g, g, d]` 网格，做 k×k、stride=k 的 average pooling，再展平成 `[B, (g//k)**2, d]` 返回。

**提示**：`g = int(N ** 0.5)`；reshape 成 `[B, g//k, k, g//k, k, d]` 后对两个 k 维取 `mean(axis=(2, 4))` 即可，无需写循环；假定 g 能被 k 整除。8 行以内。

In [ ]:
import numpy as np

def avgpool_connector(x, k):
    # TODO: [B, g*g, d] -> reshape 回网格 -> k×k 平均池化 -> [B, (g//k)**2, d]
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
import numpy as np
rng = np.random.default_rng(0)
x = rng.standard_normal((2, 576, 5))                     # 24×24 网格（ViT-L/14@336 的 patch 数）
y = avgpool_connector(x, k=2)
assert y.shape == (2, 144, 5)                            # 576 -> 144，token 数减为 1/4
assert np.allclose(y.mean(), x.mean())                   # 等块平均池化保全局均值
assert avgpool_connector(x, k=4).shape == (2, 36, 5)     # k=4：576 -> 36

tiny = np.array([1.0, 2.0, 3.0, 4.0]).reshape(1, 4, 1)   # 手算最小例：2×2 网格压成 1 个 token
assert np.allclose(avgpool_connector(tiny, k=2), 2.5)

const = np.full((1, 16, 3), 7.0)                         # 常数输入 -> 常数输出
assert np.allclose(avgpool_connector(const, k=2), 7.0)
print("✅ 练习 2 通过")

## ✏️ 练习 3：手写 Q-Former 式 cross-attention

不调 `nn.MultiheadAttention`，徒手实现**单头 cross-attention**——Q-Former"固定长度压缩"的核心机制。实现 `cross_attention(queries, feats, Wq, Wk, Wv)`：`queries: [M, d]` 是 learnable queries，`feats: [N, d]` 是图像 patch 特征，返回 `(out, attn)`，其中 `out: [M, d]`、`attn: [M, N]`。

计算流程：`Q = queries @ Wq`、`K = feats @ Wk`、`V = feats @ Wv`；`attn = softmax(Q @ K.T / sqrt(d))`（对最后一维 softmax）；`out = attn @ V`。

**提示**：softmax 先减每行最大值防上溢：`e = np.exp(s - s.max(axis=-1, keepdims=True))`，再除以 `e.sum(axis=-1, keepdims=True)`；输出 token 数只由 M 决定、与 N 无关——这正是信息瓶颈（information bottleneck）所在。10 行以内。

In [ ]:
import numpy as np

def cross_attention(queries, feats, Wq, Wk, Wv):
    # TODO: Q/K/V 投影 -> 缩放点积 + 行 softmax -> attn @ V，返回 (out, attn)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
import numpy as np
rng = np.random.default_rng(0)
M, d = 4, 8
queries = rng.standard_normal((M, d))
Wq, Wk, Wv = (rng.standard_normal((d, d)) * 0.2 for _ in range(3))

out, attn = cross_attention(queries, rng.standard_normal((12, d)), Wq, Wk, Wv)
assert out.shape == (M, d) and attn.shape == (M, 12)
assert np.allclose(attn.sum(axis=-1), 1.0)               # softmax：每个 query 的权重和为 1
assert (attn > 0).all() and (attn < 1).all()              # 权重严格落在 (0, 1)

out2, attn2 = cross_attention(queries, rng.standard_normal((49, d)), Wq, Wk, Wv)
assert out2.shape == (M, d) and attn2.shape == (M, 49)    # 固定长度压缩：N 变，输出仍是 M 个 token

same = np.tile(rng.standard_normal((1, d)), (10, 1))      # 所有 patch 相同 -> 注意力均匀 1/N
out3, attn3 = cross_attention(queries, same, Wq, Wk, Wv)
assert np.allclose(attn3, 1.0 / 10)
assert np.allclose(out3, np.tile(same[:1] @ Wv, (M, 1)))  # 输出 = 该 patch 的 V
print("✅ 练习 3 通过")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
import numpy as np

def gelu(x):
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x ** 3)))

def mlp_projector(x, W1, b1, W2, b2):
    return gelu(x @ W1 + b1) @ W2 + b2

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
import numpy as np

def avgpool_connector(x, k):
    B, N, d = x.shape
    g = int(N ** 0.5)
    grid = x.reshape(B, g, g, d)                          # 还原 2D patch 网格
    pooled = grid.reshape(B, g // k, k, g // k, k, d).mean(axis=(2, 4))
    return pooled.reshape(B, (g // k) ** 2, d)

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
import numpy as np

def cross_attention(queries, feats, Wq, Wk, Wv):
    Q, K, V = queries @ Wq, feats @ Wk, feats @ Wv
    s = Q @ K.T / np.sqrt(Q.shape[-1])                    # [M, N] 缩放点积分数
    e = np.exp(s - s.max(axis=-1, keepdims=True))         # 减行最大值防上溢
    attn = e / e.sum(axis=-1, keepdims=True)              # 行 softmax
    return attn @ V, attn

## 7. 小结 + 动手练习

**本 notebook 你亲手做到了：**
- 从零实现三种连接器：**MLP projector**（保 token 576/49）、**Q-Former 压缩器**（固定 32）、**Perceiver Resampler**（长度无关，固定 64）；
- 用真实 `CLIPVisionModel` 取 patch 特征，量化"token 数 vs 细节"权衡（49 vs 32 vs 64 对照表）；
- 复现 **stage-1 对齐的 plumbing**：冻结 `gpt2`、只训 connector，看 loss 下降，并把视觉 token 作为 `inputs_embeds` 喂入 LLM。

**动手练习：**
1. **token 数 → 成本曲线**：把 Q-Former 的 `num_query` 从 8 扫到 256，记录参数量与 cross-attn 的 FLOPs 量级，画出"token 数 vs 计算成本"曲线，对照讲解里 $O(M^2)$ 的论断。
2. **连接器消融**：把第 6 节的 connector 换成单层 `nn.Linear`（初代 LLaVA）和两层 MLP（LLaVA-1.5），在相同对齐目标下比较收敛速度——直观感受 LLaVA-1.5 升级 MLP 的动机。
3. **（进阶）真实对齐目标**：用一个真实图文对，把 connector 输出作为前缀、caption 作为目标，对冻结 gpt2 做自回归语言建模损失 `-Σ log p(w_t | w_<t, g(v))`，复现更接近真实 stage-1 的训练（仍只训 connector）。

**下一步 → 模块 05 · 视觉指令微调与对齐**：stage-2 如何解冻 LLM / 上 LoRA、用 LLaVA-Instruct 学指令遵循，以及 DPO/RLHF 偏好对齐。

---
## 🎯 真实数据胶囊题：真实 patch 特征上的 connector 投影

connector 把视觉特征维度投到 LLM 的隐藏维。用真实图像 patch，实现一个线性 connector，验证它把任意视觉维投到目标 LLM 维、形状正确，且可学习(梯度能降低对齐误差)。

> 本模块新增的**真实数据**练习：用**真实图像**把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, io, urllib.request
import numpy as np
import matplotlib.image as mpimg
CACHE=os.path.expanduser("~/.vlm_data"); os.makedirs(CACHE,exist_ok=True)
def real_image():
    "真实图像 Grace Hopper (来自 matplotlib 示例数据), 返回 (H,W,3) uint8"
    p=os.path.join(CACHE,"grace_hopper.jpg")
    if not os.path.exists(p):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/matplotlib/matplotlib/main/lib/matplotlib/mpl-data/sample_data/grace_hopper.jpg", p)
    return mpimg.imread(p)

img=real_image().astype(float)/255
P=16; H=img.shape[0]//P*P; W=img.shape[1]//P*P
raw=np.array([img[i:i+P,j:j+P].reshape(-1) for i in range(0,H,P) for j in range(0,W,P)])
# 原始像素 patch 未归一化(mean≈0.32)、各维量纲接近但尺度大，直接 GD 会发散；
# 训练前按维标准化(零均值/单位方差)是连接器训练的标准预处理，让梯度下降稳定收敛。
patches=(raw-raw.mean(0))/(raw.std(0)+1e-8)
vdim=patches.shape[1]; llm_dim=128
print(f"真实 patch 特征: {patches.shape}, 目标 LLM 维={llm_dim}  (已按维标准化: mean≈{patches.mean():.2f}, std≈{patches.std():.2f})")

**练习**：实现 `connector_forward(patches, W_proj)`（线性投影到 LLM 维）和 `connector_fit_step(patches, W_proj, target, lr)`（一步 MSE 梯度，使投影逼近 target）。

In [ ]:
def connector_forward(patches, W_proj):
    # TODO: patches @ W_proj
    raise NotImplementedError
def connector_fit_step(patches, W_proj, target, lr=1e-3):
    # TODO: pred=patches@W_proj; grad=2*patches.T@(pred-target)/n; W-=lr*grad; 返回 (W, mse)
    raise NotImplementedError


In [ ]:
# 自测
rng=np.random.default_rng(0); W_proj=rng.normal(0,0.01,(vdim,llm_dim))
out=connector_forward(patches, W_proj)
assert out.shape==(len(patches), llm_dim), "投影到 LLM 维"
# 对齐目标：视觉特征在语言空间里真实对应的嵌入。这里用一个固定线性映射+少量噪声合成出
# "标准答案"嵌入(并标准化)，连接器要学着把视觉特征投影到这个语言空间——MSE 应真实下降。
W_align=rng.normal(0,1,(vdim,llm_dim))
target=patches@W_align + 0.1*rng.normal(0,1,(len(patches),llm_dim))
target=(target-target.mean(0))/(target.std(0)+1e-8)
W=W_proj.copy(); mse0=None
for t in range(50):
    W,mse=connector_fit_step(patches, W, target, lr=1e-3)
    if t==0: mse0=mse
assert mse < mse0, f"connector 应可学习 {mse0:.2f}->{mse:.2f}"
print(f"connector ✓  形状 {out.shape}, 对齐 MSE {mse0:.2f}->{mse:.2f}")


### 📖 参考答案

In [ ]:
def connector_forward(patches, W_proj):
    return patches @ W_proj
def connector_fit_step(patches, W_proj, target, lr=1e-3):
    pred=patches@W_proj; mse=float(np.mean((pred-target)**2))
    grad=2*patches.T@(pred-target)/len(patches)
    return W_proj-lr*grad, mse
print("✓ connector(MLP/linear) 把视觉特征翻译成 LLM 能读的 token")